In [ ]:
import json
from pathlib import Path

import joblib

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

In [ ]:
BASE_DIR = Path.cwd()
DATA_PATH = BASE_DIR / "training_data.json"
MODELS_DIR = BASE_DIR / "models"

MODELS_DIR.mkdir(exist_ok=True)

In [ ]:
CATEGORY_MERGE_MAP = {
    "Education/Homework": "Education/Assignments",
    "Education/Labworks": "Education/Assignments",

    "Education/Certificates": "Documents/Personal",
    "Personal/Documents": "Documents/Personal",

    "Media/Books": "Education/Materials",
    "Education/Materials": "Education/Materials",

    "Career/Internship": "Career/Internship",
    "Career/Resume": "Career/Resume",
    "Data/Datasets": "Education/Materials",
    "Education/Project": "Education/Project",
    "Finance": "Finance",
    "IT/Licenses": "IT/Licenses",
    "Media/Transcript": "Media/Transcript",
}


def merge_category(category: str) -> str:
    return CATEGORY_MERGE_MAP.get(category, category)

In [ ]:
texts = []
labels = []

with open(DATA_PATH) as f:
    jf = json.load(f)

for note in jf:
    texts.append(note["text"])
    labels.append(note["category"])

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    texts,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

In [ ]:
pipeline = Pipeline([
    ("features", FeatureUnion([
        ("word_tfidf", TfidfVectorizer(
            analyzer="word",
            ngram_range=(1, 3),
            min_df=1,
            sublinear_tf=True,
        )),
        ("char_tfidf", TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(3, 5),
            min_df=1,
            sublinear_tf=True,
        )),
    ])),
    ("clf", LogisticRegression(
        max_iter=5000,
        class_weight="balanced",
        C=1.5,
        solver="lbfgs",
    ))
])

In [ ]:
pipeline.fit(X_train, y_train)

In [ ]:
y_pred = pipeline.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print()
print(classification_report(y_test, y_pred))

In [ ]:
MODEL_PATH = MODELS_DIR / "classifier.joblib"

joblib.dump(pipeline, MODEL_PATH)

print(f"Модель сохранена в {MODEL_PATH}")